In [1]:
import requests
import pandas as pd
import re 
import numpy as np
import os

### Load and merge datasets

In [2]:
# load first data: the UNSPSC classification of software's type
df = pd.read_excel('D:/Dropbox/Dropbox/Paper with Yao-yu/Spatial Data Programming/Data/technology_skill_onet/UNSPSC_reference.xlsx')
# only keep software skills
df = df[df['Family Title'] == 'Software']
df = df.drop(['Commodity Title'], axis = 1)

# load the second data: the software used by each occupation
df1 = pd.read_excel('D:/Dropbox/Dropbox/Paper with Yao-yu/Spatial Data Programming/Data/technology_skill_onet/technology_skills_occupation.xlsx')

In [3]:
# combine the first two datasets, so that we have a occupation title-software-software classification
df3 = pd.merge(df, df1, on = "Commodity Code")
df3 = df3[['O*NET-SOC Code',  'Title', 'Example', 'Hot Technology', 'In Demand', 'Commodity Code', 'Commodity Title', 
           'Class Code', 'Class Title', 'Family Code', 'Family Title', 'Segment Code', 'Segment Title']]
df3 = df3.sort_values(['O*NET-SOC Code', 'Example'], ascending=True)
df3

,O*NET-SOC Code,Title,Example,Hot Technology,In Demand,Commodity Code,Commodity Title,Class Code,Class Title,Family Code,Family Title,Segment Code,Segment Title
12520,11-1011.00,Chief Executives,AdSense Tracker,N,N,43232306,Data base user interface and query software,43232300,Data management and query software,43230000,Software,43000000,Information Technology Broadcasting and Teleco...
10873,11-1011.00,Chief Executives,Adobe Systems Adobe Acrobat,Y,N,43232202,Document management software,43232200,Content management software,43230000,Software,43000000,Information Technology Broadcasting and Teleco...
10821,11-1011.00,Chief Executives,Atlassian JIRA,Y,N,43232201,Content workflow software,43232200,Content management software,43230000,Software,43000000,Information Technology Broadcasting and Teleco...
11420,11-1011.00,Chief Executives,Blackbaud The Raiser's Edge,N,N,43232303,Customer relationship management CRM software,43232300,Data management and query software,43230000,Software,43000000,Information Technology Broadcasting and Teleco...
2715,11-1011.00,Chief Executives,ComputerEase construction accounting software,N,N,43231601,Accounting software,43231600,Finance accounting and enterprise resource pla...,43230000,Software,43000000,Information Technology Broadcasting and Teleco...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14936,53-7121.00,"Tank Car, Truck, and Ship Loaders",CompuWeigh GMS,N,N,43232306,Data base user interface and query software,43232300,Data management and query software,43230000,Software,43000000,Information Technology Broadcasting and Teleco...
645,53-7121.00,"Tank Car, Truck, and Ship Loaders",Distributed control system DCS,N,N,43231506,Materials requirements planning logistics and ...,43231500,Business function specific software,43230000,Software,43000000,Information Technology Broadcasting and Teleco...
29116,53-7121.00,"Tank Car, Truck, and Ship Loaders",Linux,Y,N,43233004,Operating system software,43233000,Operating environment software,43230000,Software,43000000,Information Technology Broadcasting and Teleco...
10446,53-7121.00,"Tank Car, Truck, and Ship Loaders",Microsoft Excel,Y,N,43232110,Spreadsheet software,43232100,Content authoring and editing software,43230000,Software,43000000,Information Technology Broadcasting and Teleco...


### Select data-management technology keywords:

- They should be 'Hot-technology'.
- `Class` should be data processing related: 'Class Title' is 'Finance accounting and enterprise resource planning ERP software', or 'Data management and query software', or 'Development software'
- They are in the skill set of data related jobs.
- The main list is based on Burning-glass data, we compensate with Chinese softwares that are obtained from ChatGPT and reviewed by human to check the relevancy.

In [5]:
# Keep only unique values in Name column, and based on the software titles we expand the pool to fit the case in China using ChatGPT
unique_tech = df3.drop_duplicates(subset=['Class Code', 'Class Title', 'Commodity Title', 'Example', 'O*NET-SOC Code', 'Title'])
unique_tech = unique_tech[unique_tech['Title'].str.contains('data|Data')]
# keep if 'Class Title' is 'Finance accounting and enterprise resource planning ERP software', or 'Data management and query software', or 'Development software', or 'Information exchange software'  
unique_tech = unique_tech[unique_tech['Class Title'].isin(['Finance accounting and enterprise resource planning ERP software', 'Data management and query software', 'Development software', 'Industry specific software'])] 
# keep if 'Hot Technology' == 'Y'
unique_tech = unique_tech[unique_tech['Hot Technology'] == 'Y']
unique_tech = unique_tech.loc[:, ['Commodity Title', 'Example', 'Class Code', 'Class Title']]
# keep unique values in Commodity Title
unique_tech = unique_tech.drop_duplicates(subset=['Example'])
# print the unique values in 'class title'
unique_tech['Class Title'].unique()
unique_tech

,Commodity Title,Example,Class Code,Class Title
18717,Web platform development software,AJAX,43232400,Development software
11982,Data base management system software,Amazon DynamoDB,43232300,Data management and query software
13131,Data base user interface and query software,Amazon Elastic Compute Cloud EC2,43232300,Data management and query software
13132,Data base user interface and query software,Amazon Redshift,43232300,Data management and query software
13133,Data base user interface and query software,Amazon Web Services AWS software,43232300,Data management and query software
...,...,...,...,...
15408,Object oriented data base management software,Transact-SQL,43232300,Data management and query software
17539,Object or component oriented development software,jQuery,43232400,Development software
18767,Web platform development software,React,43232400,Development software
17572,Object or component oriented development software,Objective C,43232400,Development software


In [8]:
# keep unique combination of 'Commodity Title' and 'Class Title' 
unique_tech2 = unique_tech.drop_duplicates(subset=['Commodity Title', 'Class Title'])
# keep columns 'Commodity Title' and 'Class Title'
unique_tech2 = unique_tech2.loc[:, ['Commodity Title', 'Class Title']]
# show the entire content in each column 
pd.set_option('display.max_colwidth', None)
unique_tech2

,Commodity Title,Class Title
18717,Web platform development software,Development software
11982,Data base management system software,Data management and query software
13131,Data base user interface and query software,Data management and query software
16229,Development environment software,Development software
15689,Business intelligence and data analysis software,Data management and query software
17162,Enterprise application integration software,Development software
20461,Computer aided design CAD software,Industry specific software
17537,Object or component oriented development software,Development software
15897,Configuration management software,Development software
24887,Medical software,Industry specific software


#### Manually shorten the title of the software and determine whether it is open source or developed within China (based on `unique_tech`)
- Column `Safe` is determined by human review with dummy 1 if it is open source or developed within China

In [8]:
# read a csv file (includes both Chinese and English characteristics) from 'E:/Data/job_posting/processed/title_short.csv' 
key_df = pd.read_excel('G:/Data/job_posting/processed/title_short.xlsx')
# Generating a new column to represent the combination of 'Commodity Title' and 'Safe'
key_df['Commodity_Safe'] = key_df['Commodity Title'] +  '_' + 'Safe'+  '_' + key_df['Safe'].astype(str)
# Generating a new column to represent the combination of 'Commodity Title' and 'China'
key_df['Commodity_China'] = key_df['Commodity Title'] +  '_' + 'China' + '_' + key_df['China'].astype(str)
key_df

,Commodity Title,Example,Example_short,Safe,China,Commodity_Safe,Commodity_China
0,Accounting software,Intuit QuickBooks,QuickBooks,0,0,Accounting software_Safe_0,Accounting software_China_0
1,Accounting software,速达软件,速达,1,1,Accounting software_Safe_1,Accounting software_China_1
2,Analytical or scientific software,IBM SPSS Statistics,SPSS,0,0,Analytical or scientific software_Safe_0,Analytical or scientific software_China_0
3,Analytical or scientific software,SAS,SAS,0,0,Analytical or scientific software_Safe_0,Analytical or scientific software_China_0
4,Analytical or scientific software,The MathWorks MATLAB,MATLAB,0,0,Analytical or scientific software_Safe_0,Analytical or scientific software_China_0
...,...,...,...,...,...,...,...
84,Web platform development software,Oracle JavaServer Pages JSP,JSP,0,0,Web platform development software_Safe_0,Web platform development software_China_0
85,Web platform development software,PHP,PHP,1,0,Web platform development software_Safe_1,Web platform development software_China_0
86,Web platform development software,React,React,1,0,Web platform development software_Safe_1,Web platform development software_China_0
87,Web platform development software,Ruby on Rails,Rails,1,0,Web platform development software_Safe_1,Web platform development software_China_0


In [32]:
# def classify_software(Example, api_key):
#     url = 'https://api.openai.com/v1/chat/completions'
#     headers = {'Content-Type': 'application/json',
#                'Authorization': f'Bearer {api_key}'}
#     data = {'model': 'gpt-3.5-turbo-0301',
#             'messages':[
#                 {
#                 'role': 'user', 
#                 'content': f'Is this software open source or a software developed in China "{Example}". Please answer with either YES or NO.'
#                 }
#                        ]
#             }
#     try:
#         response = requests.post(url, headers=headers, json=data, verify=False)
#         response_data = json.loads(response.text)
#         if response.status_code != 200:
#             raise Exception(response_data['error']['message'])
#         return response_data['choices'][0]['message']['content']
#     except Exception as e:
#         print(f'Error occurred: {e}')
#         return 'N/A'

In [39]:
# # Parallelize the job title classification using Python's ThreadPoolExecutor
# from concurrent.futures import ThreadPoolExecutor
# import requests
# import json
# import urllib3
# import openai

# # Disable SSL warnings
# urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# os.environ["http_proxy"] = "http://127.0.0.1:10809"
# os.environ["https_proxy"] = "http://127.0.0.1:10809"
# openai.organization = "org-VerSGqrvW53HAZizo7yd3d6Z"
# api_key = 'OPENAI_KEY_REMOVED'

# # create a thread pool with 30 worker threads
# with ThreadPoolExecutor(max_workers=30) as executor:
#     # extract the job titles and submit them to the thread pool
#     Examples = key_df['Example'].tolist()
#     futures = [executor.submit(classify_software, Example, api_key) for Example in Examples]

#     # wait for all threads to complete and get the results
#     # create an empty list to store soc codes
#     Example_short = []
#     for future in futures:
#         exp_code = future.result()
#         Example_short.append(exp_code)

#     # append the soc codes to the dataframe
#     key_df['Example_short'] = Example_short

# #key_df.to_csv('E:/Data/job_posting/processed/title_opensource.csv', index = False) 


Error occurred: HTTPSConnectionPool(host='api.openai.com', port=443): Max retries exceeded with url: /v1/chat/completions (Caused by ProxyError('Cannot connect to proxy.', FileNotFoundError(2, 'No such file or directory')))
Error occurred: HTTPSConnectionPool(host='api.openai.com', port=443): Max retries exceeded with url: /v1/chat/completions (Caused by ProxyError('Cannot connect to proxy.', FileNotFoundError(2, 'No such file or directory')))
Error occurred: HTTPSConnectionPool(host='api.openai.com', port=443): Max retries exceeded with url: /v1/chat/completions (Caused by ProxyError('Cannot connect to proxy.', FileNotFoundError(2, 'No such file or directory')))
Error occurred: HTTPSConnectionPool(host='api.openai.com', port=443): Max retries exceeded with url: /v1/chat/completions (Caused by ProxyError('Cannot connect to proxy.', FileNotFoundError(2, 'No such file or directory')))Error occurred: HTTPSConnectionPool(host='api.openai.com', port=443): Max retries exceeded with url: /v1/

### Function to check if any word from `工作描述` is present in the `software` column
- This code is run in HPC with 25 CPU cores and 5 GPUs. It took about 4 hours to finish.

In [1]:
from concurrent.futures import ProcessPoolExecutor
import pandas as pd
import os
import jieba
import cupy as cp

def process_file(filename, software_list):
    f = os.path.join(directory, filename)
    df_chunks = pd.read_csv(f, encoding="utf_8_sig", on_bad_lines='skip', usecols=['招聘主键ID', '公司ID', '工作描述'], chunksize=10000)

    output_filename = f.replace('description', 'desp_techfull')
    output_filename = os.path.splitext(output_filename)[0] + "_processed.csv"

    write_header = not os.path.exists(output_filename)
    
    jieba.enable_parallel(4)

    for chunk in df_chunks:
        new_data = cp.zeros((len(chunk), len(software_list)), dtype=cp.int32)
        
        for i, desc in enumerate(chunk['工作描述'].fillna('')):
            words = set(word.lower() for word in jieba.cut(desc, cut_all=False))
            new_data[i, :] = cp.asarray([int(software in words) for software in software_list])
        
        new_df = pd.DataFrame(cp.asnumpy(new_data), columns=software_list)
        
        chunk = pd.concat([chunk.drop(['工作描述'], axis=1).reset_index(drop=True), new_df.reset_index(drop=True)], axis=1)
        chunk.to_csv(output_filename, mode='a', index=False, encoding="utf_8_sig", header=write_header)
        
        write_header = False

    print(f"Processed {filename}")

if __name__ == "__main__":
    directory = '/share/home/320346/description/'
    filenames = [filename for filename in os.listdir(directory) if filename.startswith("job_res_")]

    key_df = pd.read_excel('/share/home/320346/title_short.xlsx')  # Change to read Excel
    key_df = key_df.drop(['Example'], axis=1).rename(columns={'Example_short': 'software'})

    software_list = key_df['software'].str.lower().unique()

    with ProcessPoolExecutor(max_workers=25) as executor:
        executor.map(process_file, filenames, [software_list] * len(filenames))

### Append the data after determining whether it is 'data programming' related, get ready for merging with firm registeration data
- This is done in the HPC with 20 CPU cores and it took about 40 mins to finish.

In [98]:
import pandas as pd
import os

path = '/share/home/320346/desp_techfull'
files = [f for f in os.listdir(path) if f.endswith('.csv')]

# Initialize an empty DataFrame to hold the final data
dfProgram = pd.DataFrame()

def add_aggregated_columns(df, key_df, classification_column):
    mapping_dict = key_df.set_index('Example_short')[classification_column].to_dict()
    mapping_dict = {k.lower(): v for k, v in mapping_dict.items()}

    # Initialize new columns with zeros
    for unique_value in key_df[classification_column].unique():
        df[unique_value] = 0

    # Summing mentions for each software and filling in the new columns
    for software_col in df.columns.difference(['招聘主键ID', '公司ID']):  # Excluding '招聘主键ID' and '公司ID'
        group_name = mapping_dict.get(software_col, None)
        if group_name:
            df[group_name] += df[software_col]

for file in files:
    file_path = os.path.join(path, file)
    
    # Error handling for file reading
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"Error reading {file}: {e}")
        continue

    df = df[df['招聘主键ID'] != '招聘ID']
    df = df.dropna(subset=['招聘主键ID'])

    df['data_software'] = (df.drop(['招聘主键ID', '公司ID'], axis=1).any(axis=1)).astype(int)

    # Add aggregated columns
    add_aggregated_columns(df, key_df, 'Commodity_Safe')
    add_aggregated_columns(df, key_df, 'Commodity_China')

    # Remove software title columns
    columns_to_remove = df.columns[df.columns.get_loc('quickbooks'):df.columns.get_loc('ajax') + 1]
    df = df.drop(columns=columns_to_remove)

    # Append to the main DataFrame
    # Consider writing to a CSV or using Dask for large datasets
    dfProgram = pd.concat([dfProgram, df], ignore_index=True)

# Summary of the resulting DataFrame
print(dfProgram.info())

dfProgram.to_csv('/share/home/320346/posting_dataprogfull.csv', index=False, encoding='utf-8')

C:\Users\DELL\AppData\Local\Temp/ipykernel_22056/2689122181.py:29: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
C:\Users\DELL\AppData\Local\Temp/ipykernel_22056/2689122181.py:29: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
C:\Users\DELL\AppData\Local\Temp/ipykernel_22056/2689122181.py:29: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
C:\Users\DELL\AppData\Local\Temp/ipykernel_22056/2689122181.py:29: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
C:\Users\DELL\AppData\Local\Temp/ipykernel_22056/2689122181.py:29: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
C:\Us

MemoryError: Unable to allocate 852. MiB for an array with shape (1, 111621527) and data type int64

#### Check the data

In [14]:
# read csv from 'G:/Data/job_posting/processed/estimation/posting_dataprogfull.csv'
dfProgram = pd.read_csv('G:/Data/job_posting/processed/estimation/posting_dataprogfull.csv', encoding='utf-8', nrows = 1000000)
dfProgram.head()

,招聘主键ID,公司ID,data_software,Accounting software_Safe_0,Accounting software_Safe_1,Analytical or scientific software_Safe_0,Business intelligence and data analysis software_Safe_1,Business intelligence and data analysis software_Safe_0,Computer aided design CAD software_Safe_0,Computer aided design CAD software_Safe_1,...,Enterprise application integration software_China_0,Enterprise resource planning ERP software_China_0,Enterprise resource planning ERP software_China_1,Medical software_China_0,Medical software_China_1,Metadata management software_China_0,Object or component oriented development software_China_0,Object oriented data base management software_China_0,Program testing software_China_0,Web platform development software_China_0
0,110327934,56272790,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,110327935,56272790,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,110327936,64762996,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,110327937,1013142,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,110327938,64762996,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
